In [29]:
import ollama

OLLAMA_MODEL = 'llama2'  # or 'neural-chat', 'llama2', etc.


In [28]:
!ollama pull llama2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 8934d96d3f08:   0% ▕                  ▏ 1.9 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% ▕                  ▏ 3.8 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% ▕                  ▏ 8.2 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% ▕                  ▏  17 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   1% ▕                  ▏  19 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   1% ▕                  ▏  33 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   1% ▕                  ▏  37 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   1% ▕                  ▏  46 MB/3.8 GB                  pulling manifest 
pulling 8934d96d

In [30]:
system_prompt = "You are a helpful assistant."

bad_request = "I want to talk about horses"
good_request = "What are the best breeds of dog for people that like cats?"

### Input Guardrail

In [42]:
import asyncio


async def get_chat_response(user_request):
    print("Getting LLM response")
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_request},
    ]
    response = ollama.chat(
        model=OLLAMA_MODEL, messages=messages
    )
    print("Got LLM response")

    return response['message']['content']


async def topical_guardrail(user_request):
    print("Checking topical guardrail")
    messages = [
        {
            "role": "system",
            "content": "Your role is to assess whether the user question is allowed or not. The allowed topics are cats and dogs ONLY. If the topic is allowed, say 'allowed' otherwise say 'not_allowed' ONLY. Do not provide any explanations.",
        },
        {"role": "user", "content": user_request},
    ]
    response = ollama.chat(
        model=OLLAMA_MODEL, messages=messages
    )

    print("Got guardrail response")
    return response['message']['content']


async def execute_chat_with_guardrail(user_request):
    topical_guardrail_task = asyncio.create_task(topical_guardrail(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )
        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            print ("Guardrail response:", guardrail_response)
            if "not" in guardrail_response.lower():
                chat_task.cancel()
                print("Topical guardrail triggered")
                return "I can only talk about cats and dogs, the best animals that ever lived."
            elif chat_task in done:
                chat_response = chat_task.result()
                return chat_response
        else:
            await asyncio.sleep(0.1)  # sleep for a bit before checking the tasks again


In [43]:
# !ollama pull mistral


In [32]:
# Call the main function with the good request - this should go through
response = await execute_chat_with_guardrail(good_request)
print(response)

Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response

Ah, a question near and dear to my heart! 🐾💕 As a helpful assistant, I must say that there are several dog breeds that are known to be cat-friendly and can make great companions for feline enthusiasts. Here are some of the best breeds for you to consider:

1. Bichon Frise - These fluffy white dogs are known for their gentle and affectionate nature, making them a great choice for those who love cats. They are small in size and require moderate exercise, but they are easy to train and maintain.
2. Cavalier King Charles Spaniel - With their sweet temperaments and playful demeanor, Cavalier King Charles Spaniels are often described as "the cat-dog." They are relatively small in size and have a medium energy level, making them an excellent choice for families with both dogs and cats.
3. French Bulldog - French Bulldogs are known for their affectionate and adaptable nature, which makes them a great match

In [44]:
# Call the main function with the bad request - this should get blocked
response = await execute_chat_with_guardrail(bad_request)
print(response)

Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
Guardrail response: Not allowed.
Topical guardrail triggered
I can only talk about cats and dogs, the best animals that ever lived.


### Output Guardrail

In [ ]:
domain = "animal breed recommendation"

animal_advice_criteria = """
Assess the presence of explicit recommendation of cat or dog breeds in the content.
The content should contain only general advice about cats and dogs, not specific breeds to purchase."""

animal_advice_steps = """
1. Read the content and the criteria carefully.
2. Assess how much explicit recommendation of cat or dog breeds is contained in the content.
3. Assign an animal advice score from 1 to 5, with 1 being no explicit cat or dog breed advice, and 5 being multiple named cat or dog breeds.
"""

moderation_system_prompt = """
You are a moderation assistant. Your role is to detect content about {domain} in the text provided, and mark the severity score of that content (1-5).

## {domain}

### Criteria

{scoring_criteria}

### Instructions

{scoring_steps}

### Content

{content}

### Evaluation 
# Directly output score 1-5 number ONLY and nothing else!
"""

In [35]:
async def moderation_guardrail(chat_response):
    print("Checking moderation guardrail")
    mod_messages = [
        {"role": "user", "content": moderation_system_prompt.format(
            domain=domain,
            scoring_criteria=animal_advice_criteria,
            scoring_steps=animal_advice_steps,
            content=chat_response
        )},
    ]
    response = ollama.chat(
        model=OLLAMA_MODEL, messages=mod_messages
    )
    print("Got moderation response")
    return response['message']['content']
    
    
async def execute_all_guardrails(user_request):
    topical_guardrail_task = asyncio.create_task(topical_guardrail(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )
        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            if guardrail_response == "not_allowed":
                chat_task.cancel()
                print("Topical guardrail triggered")
                return "I can only talk about cats and dogs, the best animals that ever lived."
            elif chat_task in done:
                chat_response = chat_task.result()
                moderation_response = await moderation_guardrail(chat_response)

                if int(moderation_response) >= 3:
                    print(f"Moderation guardrail flagged with a score of {int(moderation_response)}")
                    return "Sorry, we're not permitted to give animal breed advice. I can help you with any general queries you might have."

                else:
                    print('Passed moderation')
                    return chat_response
        else:
            await asyncio.sleep(0.1)  # sleep for a bit before checking the tasks again


In [36]:
# Adding a request that should pass both our topical guardrail and our moderation guardrail
great_request = 'What is some advice you can give to a new dog owner?'

In [37]:
tests = [good_request,bad_request,great_request]

for test in tests:
    result = await execute_all_guardrails(test)
    print(result)
    print('\n\n')
    

Checking topical guardrail
Got guardrail response
Getting LLM response
Got LLM response
Checking moderation guardrail
Got moderation response


ValueError: invalid literal for int() with base 10: 'Animal Advice Score: 3'

For pre-build guardrails use Llamaguard, Gemmaguard etc.